In [17]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:14pt;}
div.output {font-size:14pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:10pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:10pt;padding:5px;}
table.dataframe{font-size:15px;
</style>
"""))

# OpenAI를 활용한 생성형 AI
## K-디지털트레이닝) 기업 맞춤형 AI+X 융복합 인재 양성과정
### 생성형 ai 랭체인 구현
- 원하는 모델을 이용하여 나라를 입력하면 해당 나라의 가장 유명한 음식을 추천하는 렝체인을 구현(food_chain))
- 음식을 입력하면 레시피를 생성하는 렝체인을 구현(recipe_chain)
- 위의 두 체인을 연결하여, 나라이름을 입력받아, 해당 나라의 가장 유명한 음식의 레시피를 생성하는 렝체인 프로그램을 구현하시오.


In [19]:
import os
import streamlit as st
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.chains import LLMChain, SequentialChain

load_dotenv()

llm = ChatOpenAI(
    openai_api_key=os.environ.get("OPENAI_API_KEY"), model_name="gpt-4o-mini")

# 음식 추천
food_prompt_template = ChatPromptTemplate.from_template(
    """당신은 세계 최고의 특급 요리사입니다.
    '{country}' 국가를 대표하는 가장 유명한 음식 이름을 1개만 추천해주세요.
    음식 이름 한 단어만 답변하고 다른 부연 설명은 절대 하지 마세요."""
)

food_chain = LLMChain(
    llm=llm,
    prompt=food_prompt_template,
    output_key="food"
)

# 레시피
recipe_prompt_template = ChatPromptTemplate.from_template(
    """당신은 세계 최고의 특급 요리사입니다.
    '{food}' 음식을 맛있게 만들 수 있는 레시피를 알려주세요.
    답변 형식은 아래와 같이 답변해주세요.

    ** 재료: **
    1. [재료 1]
    2. [재료 2]
    3. ...

    ** 만드는 법: **
    1. [과정 1]
    2. [과정 2]
    3. ...
    """
)

recipe_chain = LLMChain(
    llm=llm,
    prompt=recipe_prompt_template,
    output_key="recipe"
)

# 두 체인 연결
full_chain = SequentialChain(
    chains=[food_chain, recipe_chain],
    input_variables=["country"],
    output_variables=["food", "recipe"],
    verbose=True
)

# --- 프로그램 실행 ---
print("\n--- 프로그램 실행 ---")
country_name = "멕시코"
final_result = full_chain.invoke({"country": country_name})

print("\n\n--- 결과 ---")
print(f"입력 국가 : {country_name}")
print(f"추천 음식 : {final_result['food']}")
print("\n[생성 레시피]")
print(final_result['recipe'])


--- 프로그램 실행 ---


> Entering new SequentialChain chain...

> Finished chain.


--- 결과 ---
입력 국가 : 멕시코
추천 음식 : 타코

[생성 레시피]
** 재료: **
1. 타코 쉘 (또는 밀가루 토르티야)
2. 다진 소고기 (또는 닭고기, 돼지고기)
3. 양파 1개 (다진 것)
4. 마늘 2쪽 (다진 것)
5. 칠리 파우더 1큰술
6. 커민 가루 1작은술
7. 소금과 후추 (기호에 따라)
8. 양상추 (잘게 썬 것)
9. 토마토 (다진 것)
10. 체다 치즈 (갈은 것)
11. 아보카도 (슬라이스 한 것)
12. 사워크림 (선택 사항)
13. 고수 (선택 사항)

** 만드는 법: **
1. 팬에 올리브유를 두르고 중간 불에서 다진 양파와 마늘을 넣고 볶아 향이 올라올 때까지 조리합니다.
2. 다진 소고기를 팬에 추가하고 고기가 갈색이 될 때까지 볶습니다.
3. 칠리 파우더, 커민 가루, 소금, 후추를 넣고 잘 섞은 후 5분 정도 더 요리합니다.
4. 타코 쉘 또는 밀가루 토르티야를 따뜻하게 데웁니다.
5. 타코 쉘 안에 볶은 고기 혼합물을 적당량 넣습니다.
6. 잘게 썬 양상추, diced 토마토, 갈은 체다 치즈, 아보카도를 적층하여 넣습니다.
7. 원한다면 사워크림과 신선한 고수를 위에 얹어 마무리합니다.
8. 클래식한 타코를 소스와 함께 서빙하고, 즐깁니다! 

맛있게 드세요!
